# ARC Colab Agent

Run this notebook in Google Colab to turn this runtime into an ARC worker.

1. Set `ARC_URL` and `ARC_API_KEY` below (your ARC app URL and its API key).
2. Runtime → Run all. The cell keeps polling ARC for jobs.
3. Anything you send ARC that starts with "run on colab..." gets queued; this runtime picks it up, executes the Python, and posts the result back to ARC.
4. Keep this tab open — the worker shows ONLINE in ARC's capability matrix while the cell runs (Colab free tier gives you a few hours; it survives disconnects for a while, just re-run when it stops).

Notebook -> ARC connection is outbound-only (ARC never reaches into Colab — that's a Colab platform rule, not a choice).

In [ ]:
import json, time, uuid, sys, io, traceback, requests
from IPython.display import clear_output

ARC_URL = "https://arc-api-d151.onrender.com"   # <-- your ARC app
ARC_API_KEY = ""                                 # <-- your ARC API key

H = {"Authorization": f"Bearer {ARC_API_KEY}"} if ARC_API_KEY else {}
GPU = ""
try:
    import torch
    GPU = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    GPU = "cpu"

r = requests.post(f"{ARC_URL}/colab", json={"action": "register", "name": f"colab-{uuid.uuid4().hex[:4]}", "gpu": GPU}, headers=H, timeout=30)
r.raise_for_status()
WORKER = r.json()["id"]
print(f"ARC Colab worker online: {WORKER} (gpu={GPU})")

while True:
    try:
        requests.post(f"{ARC_URL}/colab", json={"action": "heartbeat", "worker_id": WORKER}, headers=H, timeout=30)
        job = requests.post(f"{ARC_URL}/colab", json={"action": "next", "worker_id": WORKER}, headers=H, timeout=30).json().get("job")
        if job:
            print(f">> claimed job {job['id']}: {job['title']}")
            ok, out = True, ""
            buf = io.StringIO()
            old = sys.stdout
            try:
                sys.stdout = buf
                exec(compile(job["code"], f"arc-job-{job['id']}", "exec"), {})
            except Exception:
                ok = False
                out = traceback.format_exc()
            finally:
                sys.stdout = old
                out = (buf.getvalue() + "\n" + out).strip()[:20000]
            requests.post(f"{ARC_URL}/colab", json={"action": "result", "worker_id": WORKER, "job_id": job["id"], "ok": ok, "output": out}, headers=H, timeout=60)
            print(("OK\n" if ok else "FAILED\n") + out[:500] + "\n" + "-"*60)
        time.sleep(10)
    except KeyboardInterrupt:
        print("worker stopped"); break
    except Exception as e:
        print("poll error:", e); time.sleep(15)
